# Lab 9.1 — Pipeline Medallion como Databricks Workflow

**Sesión 9 | Módulo 4: Productionizing Data Pipelines**

## Objetivo
Construir un pipeline Bronze → Silver → Gold usando `transacciones_retail.csv` y configurarlo como un **Databricks Workflow** de 3 tareas con dependencias secuenciales.

## Arquitectura
```
transacciones_retail.csv
         |
   [vol_landing/sesion09/]
         |
   TAREA 1: bronze_load  (este notebook, sección Lab 9B)
         |
   dbassociate.bronze.transacciones_retail_raw
         |
   TAREA 2: silver_transform  (este notebook, sección Lab 9C)
         |
   dbassociate.silver.transacciones_clean
         |
   TAREA 3: gold_aggregation  (este notebook, sección Lab 9D)
         |
   dbassociate.gold.ventas_por_tienda
```

## Cómo usar este notebook
- **Modo standalone:** ejecutar celda por celda de arriba hacia abajo para probar el pipeline completo
- **Modo Workflow:** copiar cada sección (9B, 9C, 9D) a notebooks separados y crear un Job con 3 tareas

**Runtime requerido:** DBR 13.3 LTS o superior | **Catálogo:** dbassociate con Unity Catalog

## Lab 9A — Verificación y configuración base

In [0]:
# Verificar que el archivo fuente está en el Volume
files = dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion09/")
display(files)

In [0]:
# Constantes del laboratorio — usar en todos los notebooks de tareas
CATALOG       = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing"
SOURCE_PATH   = f"{VOLUME_PATH}/sesion09"

BRONZE_TABLE  = f"{CATALOG}.{SCHEMA_BRONZE}.transacciones_retail_raw"
SILVER_TABLE  = f"{CATALOG}.{SCHEMA_SILVER}.transacciones_clean"
GOLD_TABLE    = f"{CATALOG}.{SCHEMA_GOLD}.ventas_por_tienda"

print(f"Source  : {SOURCE_PATH}")
print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Gold    : {GOLD_TABLE}")

## Lab 9B — TAREA 1: Ingesta Bronze

Esta sección corresponde a la primera tarea del Workflow (`bronze_load`).

**Responsabilidades de esta tarea:**
- Leer el CSV con schema explícito (nunca `inferSchema=True` en producción)
- Agregar columnas técnicas de auditoría
- Escribir la tabla Bronze en Delta
- Exponer el count de registros via **Task Values** para la tarea siguiente

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.functions import current_timestamp, col, lit

schema_transacciones = StructType([
    StructField("order_id",      StringType(),  False),
    StructField("fecha",         DateType(),    True),
    StructField("tienda_id",     StringType(),  True),
    StructField("cliente_id",    StringType(),  True),
    StructField("producto",      StringType(),  True),
    StructField("categoria",     StringType(),  True),
    StructField("cantidad",      IntegerType(), True),
    StructField("precio_unit",   DoubleType(),  True),
    StructField("descuento_pct", DoubleType(),  True),
    StructField("total",         DoubleType(),  True),
    StructField("metodo_pago",   StringType(),  True),
    StructField("estado",        StringType(),  True),
])

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_transacciones)
    .load(f"{SOURCE_PATH}/transacciones_retail.csv")
)

# Columnas técnicas de auditoría Bronze
df_bronze = (
    df_raw
    .withColumn("_ingestion_ts",  current_timestamp())
    .withColumn("_source_file",   col("_metadata.file_name"))
    .withColumn("_batch_session", lit("sesion09"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

bronze_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze cargado: {bronze_count} registros -> {BRONZE_TABLE}")

# Task Value: disponible para la tarea silver_transform en el Workflow
dbutils.jobs.taskValues.set(key="bronze_count", value=bronze_count)

In [0]:
# Inspeccionar la tabla Bronze
display(spark.table(BRONZE_TABLE).limit(5))

## Lab 9C — TAREA 2: Transformación Silver

Esta sección corresponde a la segunda tarea del Workflow (`silver_transform`).

**Responsabilidades de esta tarea:**
- Leer el Task Value de la tarea anterior para validación
- Aplicar reglas de calidad: filtrar nulos, normalizar strings
- Calcular columnas derivadas
- Excluir registros en estado DEVUELTO (se procesan en pipeline separado)
- Exponer el count de registros válidos via Task Values

In [0]:
from pyspark.sql.functions import col, upper, trim, when, round as spark_round

# Leer Task Value de bronze_load
# debugValue se usa cuando se ejecuta el notebook fuera del Workflow
bronze_count = dbutils.jobs.taskValues.get(
    taskKey="bronze_load",
    key="bronze_count",
    default=0,
    debugValue=35
)
print(f"Registros recibidos desde bronze: {bronze_count}")

df_bronze = spark.table(BRONZE_TABLE)

df_silver = (
    df_bronze
    # Reglas de calidad
    .filter(col("order_id").isNotNull())
    .filter(col("total") > 0)
    # Normalización de strings
    .withColumn("categoria",   upper(trim(col("categoria"))))
    .withColumn("estado",      upper(trim(col("estado"))))
    .withColumn("metodo_pago", upper(trim(col("metodo_pago"))))
    # Columnas derivadas
    .withColumn("tiene_descuento",
        when(col("descuento_pct") > 0, True).otherwise(False))
    .withColumn("monto_descuento",
        spark_round(
            col("precio_unit") * col("cantidad") * col("descuento_pct") / 100, 2))
    # Solo transacciones activas en Silver
    .filter(col("estado").isin("COMPLETADO", "PENDIENTE"))
    # Limpiar columnas técnicas de Bronze
    .drop("_source_file", "_batch_session")
    .withColumnRenamed("_ingestion_ts", "bronze_ts")
)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

silver_count = spark.table(SILVER_TABLE).count()
print(f"Silver generado: {silver_count} registros validos -> {SILVER_TABLE}")
print(f"Registros excluidos (devueltos/invalidos): {bronze_count - silver_count}")

dbutils.jobs.taskValues.set(key="silver_count", value=silver_count)

## Lab 9D — TAREA 3: Agregación Gold

Esta sección corresponde a la tercera tarea del Workflow (`gold_aggregation`).

**Responsabilidades de esta tarea:**
- Leer el Task Value de silver_transform
- Calcular KPIs de ventas agrupados por tienda y categoría
- Escribir la tabla Gold con métricas listas para consumo BI

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg,
    max as spark_max, round as spark_round
)

silver_count = dbutils.jobs.taskValues.get(
    taskKey="silver_transform",
    key="silver_count",
    default=0,
    debugValue=33
)
print(f"Procesando {silver_count} registros desde silver")

df_silver = spark.table(SILVER_TABLE)

df_gold = (
    df_silver
    .groupBy("tienda_id", "categoria")
    .agg(
        count("order_id").alias("num_transacciones"),
        spark_round(spark_sum("total"), 2).alias("ingresos_total"),
        spark_round(avg("total"), 2).alias("ticket_promedio"),
        spark_sum("cantidad").alias("unidades_vendidas"),
        spark_max("fecha").alias("ultima_venta")
    )
)

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

gold_count = spark.table(GOLD_TABLE).count()
print(f"Gold generado: {gold_count} filas (combinaciones tienda x categoria) -> {GOLD_TABLE}")
display(spark.table(GOLD_TABLE).orderBy("tienda_id", "categoria"))

## Lab 9E — Crear el Workflow en la UI de Databricks

Ahora que el código funciona en modo standalone, vamos a orquestarlo como un Workflow.

### Paso 1: Preparar los notebooks de tareas
En producción, cada tarea debería ser un notebook separado. Para este lab, las secciones 9B, 9C y 9D de este notebook representan las 3 tareas.

### Paso 2: Crear el Job
1. Ir al menú lateral → **Workflows**
2. Clic en **Create job**
3. Nombre: `sesion09_pipeline_retail`

### Paso 3: Configurar las tareas

**Tarea 1 — bronze_load:**
- Task name: `bronze_load`
- Type: `Notebook`
- Path: ruta a este notebook (o notebook separado con el código de Lab 9B)
- Compute: `Create new job cluster` → DBR 13.3 LTS, 1 worker, instancia Standard_DS3_v2

**Tarea 2 — silver_transform:**
- Task name: `silver_transform`
- Type: `Notebook`
- Depends on: `bronze_load` ← esta es la dependencia clave
- Compute: mismo job cluster

**Tarea 3 — gold_aggregation:**
- Task name: `gold_aggregation`
- Type: `Notebook`
- Depends on: `silver_transform`
- Compute: mismo job cluster

### Paso 4: Configurar schedule y alertas
- Clic en **Add schedule** → Every day at 02:00 AM
- Clic en el ícono de campana → Add email notification on failure

### Paso 5: Ejecutar
- Clic en **Run now**
- Observar el DAG en la UI: cada tarea cambia de estado pending → running → success
- Explorar los logs de cada tarea haciendo clic sobre ella

### Puntos clave a observar:
- El cluster se crea al inicio del run y se destruye al terminar (job cluster)
- Los Task Values aparecen en el panel de la tarea downstream
- La duración de cada tarea se muestra en el timeline del run

## Limpieza

In [0]:
# Ejecutar solo al terminar el laboratorio
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
print("Tablas eliminadas correctamente")